In [43]:
import json
import pickle
from tensorflow.keras.models import load_model

# Load dataset (intents)
with open("commands.json", "r") as f:
    intents = json.load(f)

# Load vocabulary + classes
with open("words.pkl", "rb") as f:
    words = pickle.load(f)

with open("classes.pkl", "rb") as f:
    classes = pickle.load(f)

# Load model
model = load_model("chatbot_model.h5")

In [44]:
import nltk

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to /Users/gres1/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/gres1/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/gres1/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [45]:
def clean_up_sentence(sentence):
    # tokenize 
    sentence_words = nltk.word_tokenize(sentence)
    # stemmming
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words

In [46]:
# return bag of words array

def bow(sentence, words, show_details=True):
    # tokenize the pattern
    sentence_words = clean_up_sentence(sentence)
    # bag of words
    bag = [0]*len(words)  
    for s in sentence_words:
        for i,w in enumerate(words):
            if w == s: 
                
                bag[i] = 1
                if show_details:
                    print ("found in bag: %s" % w)
    return(np.array(bag))

In [47]:
def predict_class(sentence, model):
    # filter out predictions based on below threshold
    p = bow(sentence, words,show_details=False)
    res = model.predict(np.array([p]))[0]
    ERROR_THRESHOLD = 0.25
    results = [[i,r] for i,r in enumerate(res) if r>ERROR_THRESHOLD]
    # sort by strength of probability
    results.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in results:
        return_list.append({"intent": classes[r[0]], "probability": str(r[1])})
    return return_list

In [48]:
def getResponse(ints, intents_json):
    tag = ints[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if(i['tag']== tag):
            result = random.choice(i['responses'])
            break
    return result

In [50]:
def chatbot_response(msg):
    ints = predict_class(msg, model)
    res = getResponse(ints, intents)
    return res

In [52]:
import numpy as np

def predict_class(sentence, model):
    # 1) bag-of-words vector (length 88)
    p = bow(sentence, words, show_details=False)

    # 2) make batch (1, 88)
    x = np.array([p], dtype=np.float32)

    # 3) add time-step dimension -> (1, 1, 88)
    x = np.expand_dims(x, axis=1)

    # 4) predict -> usually (1, 1, 9) or (1, 9)
    probs = model.predict(x, verbose=0)

    # 5) squeeze all singleton dims -> (9,)
    probs = np.squeeze(probs)

    # safety: make sure it’s 1D
    if probs.ndim != 1:
        probs = probs.reshape(-1)

    ERROR_THRESHOLD = 0.25

    results = []
    for i, prob in enumerate(probs):
        prob = float(prob)  # convert numpy scalar -> normal Python float
        if prob > ERROR_THRESHOLD:
            results.append([i, prob])

    results.sort(key=lambda x: x[1], reverse=True)

    return [{"intent": classes[i], "probability": str(prob)} for i, prob in results]

In [53]:
import speech_recognition as sr

r = sr.Recognizer()

while True:
    with sr.Microphone() as source:
        # 1) calibrate to the room noise for 0.8s
        r.adjust_for_ambient_noise(source, duration=0.8)

        print("Speak (or say 'exit')...")
        try:
            # 2) wait max 3 seconds for you to START speaking
            audio = r.listen(source, timeout=3, phrase_time_limit=5)
        except sr.WaitTimeoutError:
            print("No speech started (timeout). Try again.\n")
            continue

    try:
        text = r.recognize_google(audio, language="en-US")
        print("You:", text)

        if text.lower() in {"exit", "quit", "stop"}:
            break

        res = chatbot_response(text)
        print("Bot:", res, "\n")

    except sr.UnknownValueError:
        print("Speech heard, but couldn't decode words. Try again.\n")
    except sr.RequestError as e:
        print("Google STT failed (internet/API):", e, "\n")
    except Exception as e:
        print("Chatbot crashed (not audio):", repr(e), "\n")

Speak (or say 'exit')...
You: hello
Bot: Hello, thanks for asking 

Speak (or say 'exit')...
You: nice you're working
Bot: Have a nice day 

Speak (or say 'exit')...
Speech heard, but couldn't decode words. Try again.

Speak (or say 'exit')...
You: exit


In [19]:
import traceback

try:
    print("Bot:", chatbot_response("hello"))
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/var/folders/z0/rpp8sh914qv0x1_qpd65k81r0000gn/T/ipykernel_51335/2419035594.py", line 4, in <module>
    print("Bot:", chatbot_response("hello"))
                  ~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/var/folders/z0/rpp8sh914qv0x1_qpd65k81r0000gn/T/ipykernel_51335/2302567536.py", line 2, in chatbot_response
    ints = predict_class(msg, model)
  File "/var/folders/z0/rpp8sh914qv0x1_qpd65k81r0000gn/T/ipykernel_51335/2042171648.py", line 3, in predict_class
    p = bow(sentence, words,show_details=False)
  File "/var/folders/z0/rpp8sh914qv0x1_qpd65k81r0000gn/T/ipykernel_51335/3951542265.py", line 5, in bow
    sentence_words = clean_up_sentence(sentence)
  File "/var/folders/z0/rpp8sh914qv0x1_qpd65k81r0000gn/T/ipykernel_51335/1330134309.py", line 3, in clean_up_sentence
    sentence_words = nltk.word_tokenize(sentence)
  File "/Users/gres1/Downloads/1767097362_datasetsanddependencyfiles/.venv/lib/python3.13/site-packages/nltk/tokenize/__init__.py"

In [20]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /Users/gres1/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [33]:
import numpy as np

x = np.array([bow("hello", words, show_details=False)], dtype=np.float32)  # (1, 88)
x = np.expand_dims(x, axis=1)  # (1, 1, 88)  <-- this matches (None, None, 88)

pred = model.predict(x, verbose=0)
pred

array([[[2.6994262e-06, 2.3828430e-05, 1.3127591e-06, 5.1030605e-05,
         9.9985909e-01, 2.3625935e-05, 1.6230049e-05, 1.0565506e-05,
         1.1597422e-05]]], dtype=float32)